<a href="https://colab.research.google.com/github/IrynaSukhoverkhova/my-first-work-place/blob/main/HW_15_4_%D0%90%D0%BD%D0%B0%D0%BB%D1%96%D0%B7_%D0%90_%D0%92_%D1%82%D0%B5%D1%81%D1%82%D1%96%D0%B2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Аналіз A/B-тестів

Ви - аналітик даних в ІТ-компанії і до вас надійшла задача проаналізувати дані A/B тесту в популярній [грі Cookie Cats](https://www.facebook.com/cookiecatsgame). Це - гра-головоломка в стилі «з’єднай три», де гравець повинен з’єднати плитки одного кольору, щоб очистити дошку та виграти рівень. На дошці також зображені співаючі котики :)

Під час проходження гри гравці стикаються з воротами, які змушують їх чекати деякий час, перш ніж вони зможуть прогресувати або зробити покупку в додатку.

У цьому блоці завдань ми проаналізуємо результати A/B тесту, коли перші ворота в Cookie Cats було переміщено з рівня 30 на рівень 40. Зокрема, ми хочемо зрозуміти, як це вплинуло на утримання (retention) гравців. Тобто хочемо зрозуміти, чи переміщення воріт на 10 рівнів пізніше якимось чином вплинуло на те, що користувачі перестають грати в гру раніше чи пізніше з точки зору кількості їх днів з моменту встановлення гри.

Будемо працювати з даними з файлу `cookie_cats.csv`. Колонки в даних наступні:

- `userid` - унікальний номер, який ідентифікує кожного гравця.
- `version` - чи потрапив гравець в контрольну групу (gate_30 - ворота на 30 рівні) чи тестову групу (gate_40 - ворота на 40 рівні).
- `sum_gamerounds` - кількість ігрових раундів, зіграних гравцем протягом першого тижня після встановлення
- `retention_1` - чи через 1 день після встановлення гравець повернувся і почав грати?
- `retention_7` - чи через 7 днів після встановлення гравець повернувся і почав грати?

Коли гравець встановлював гру, його випадковим чином призначали до групи gate_30 або gate_40.

1. Для початку, уявімо, що ми тільки плануємо проведення зазначеного А/B-тесту і хочемо зрозуміти, дані про скількох користувачів нам треба зібрати, аби досягнути відчутного ефекту. Відчутним ефектом ми вважатимемо збільшення утримання на 1% після внесення зміни. Обчисліть, скільки користувачів сумарно нам треба аби досягнути такого ефекту, якщо продакт менеджер нам повідомив, що базове утримання є 19%.

In [2]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.stats.api as sms
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from math import ceil

%matplotlib inline

In [3]:
effect_size = sms.proportion_effectsize(0.2, 0.19)    # Розрахунок розміру ефекту на основі наших очікуваних показників

In [4]:
effect_size

np.float64(0.025241594409087353)

In [8]:
required_n = sms.NormalIndPower().solve_power(
    effect_size,
    power=0.8,
    alpha=0.05,
    ratio=1
    )                                                  # Розрахунок необхідного розміру вибірки

required_n = ceil(required_n)                          # Округлення до наступного цілого числа

# Округлюємо в більшу сторону та множимо на 2 (для обох груп)
total_sample_size = int(required_n * 2)
print(f"Необхідна кількість користувачів на одну групу: {int(required_n)}")
print(f"Сумарна кількість користувачів для тесту: {total_sample_size}")

Необхідна кількість користувачів на одну групу: 24638
Сумарна кількість користувачів для тесту: 49276


2. Зчитайте дані АВ тесту у змінну `df` та виведіть середнє значення показника показник `retention_7` (утримання на 7 день) по версіям гри. Сформулюйте гіпотезу: яка версія дає краще утримання через 7 днів після встановлення гри?

In [12]:
df = pd.read_csv('cookie_cats.csv', sep=';')

In [13]:
print(df.columns)

Index(['userid', 'version', 'sum_gamerounds', 'retention_1', 'retention_7'], dtype='object')


In [14]:
retention_7_grouped = df.groupby('version')['retention_7'].mean()
print(retention_7_grouped)

version
gate_30    0.190201
gate_40    0.182000
Name: retention_7, dtype: float64


**Формулювання гіпотез:**

1. **Нульова гіпотеза ($H_0$)**: між утриманням на 7-й день (retention_7) для контрольної групи (gate_30) та тестової групи (gate_40) немає статистично значущої різниці.
2. **Альтернативна гіпотеза ($H_1$):** існує статистично значуща різниця в утриманні гравців на 7-й день між версіями гри з воротами на 30-му та 40-му рівнях.

**Яка версія дає краще утримання:**

Краще утримання через 7 днів після встановлення гри дає версія gate_30 (ворота на 30-му рівні). Згідно з розрахунками, її показник становить ~19.02%, тоді як у версії gate_40 він нижчий і дорівнює ~18.20%. Тобто перенесення воріт вище (на 40-й рівень) погіршує довгострокове утримання гравців.

3. Перевірте з допомогою пасуючого варіанту z-тесту, чи дає якась з версій гри кращий показник `retention_7` на рівні значущості 0.05. Обчисліть також довірчі інтервали для варіантів до переміщення воріт і після. Виведіть результат у форматі:

    ```
    z statistic: ...
    p-value: ...
    Довірчий інтервал 95% для групи control: [..., ...]
    Довірчий інтервал 95% для групи treatment: [..., ...]
    ```

    де замість `...` - обчислені значення.
    
    В якості висновку дайте відповідь на два питання:  

      1. Чи є статистична значущою різниця між поведінкою користувачів у різних версіях гри?   
      2. Чи перетинаються довірчі інтервали утримання користувачів з різних версій гри? Про що це каже?  


In [15]:
from statsmodels.stats.proportion import proportions_ztest, proportion_confint

# Кількість успіхів (гравці, які повернулися на 7-й день) та загальна кількість гравців у групах
counts = [
    df[df['version'] == 'gate_30']['retention_7'].sum(),
    df[df['version'] == 'gate_40']['retention_7'].sum()
]
nobs = [
    df[df['version'] == 'gate_30']['retention_7'].count(),
    df[df['version'] == 'gate_40']['retention_7'].count()
]

# Z-тест
z_stat, p_val = proportions_ztest(counts, nobs)

# Довірчі інтервали 95% (alpha = 0.05)
ci_gate_30 = proportion_confint(counts[0], nobs[0], alpha=0.05, method='normal')
ci_gate_40 = proportion_confint(counts[1], nobs[1], alpha=0.05, method='normal')

# Виведення результатів у потрібному форматі
print(f"z statistic: {z_stat:.4f}")
print(f"p-value: {p_val:.4f}")
print(f"Довірчий інтервал 95% для групи control: [{ci_gate_30[0]:.4f}, {ci_gate_30[1]:.4f}]")
print(f"Довірчий інтервал 95% для групи treatment: [{ci_gate_40[0]:.4f}, {ci_gate_40[1]:.4f}]")

z statistic: 3.1644
p-value: 0.0016
Довірчий інтервал 95% для групи control: [0.1866, 0.1938]
Довірчий інтервал 95% для групи treatment: [0.1785, 0.1855]


1. **Чи є статистично значущою різниця між поведінкою користувачів у різних версіях гри?**

Так, різниця є статистично значущою. Оскільки отримане значення $p\text{-value}$ ($0.0016$) є значно меншим за обраний рівень значущості $\alpha = 0.05$, ми відхиляємо нульову гіпотезу. Це означає, що перенесення воріт з 30-го на 40-й рівень статистично значуще вплинуло на утримання гравців (retention_7).

2. **Чи перетинаються довірчі інтервали утримання користувачів з різних версій гри? Про що це каже?**

Ні, 95% довірчі інтервали для контрольної групи ([0.1866, 0.1938]) та тестової групи ([0.1785, 0.1855]) не перетинаються. Верхня межа тестової групи ($0.1855$) є нижчою за нижню межу контрольної групи ($0.1866$). Це додатково підтверджує висновок z-тесту: версія з воротами на 30-му рівні (gate_30) демонструє стабільно краще утримання гравців на 7-й день, і ця різниця не є випадковою.

4. Виконайте тест Хі-квадрат на рівні значущості 5% аби визначити, чи є залежність між версією гри та утриманням гравця на 7ий день після реєстрації.

    - Напишіть, як для цього тесту будуть сформульовані гіпотези.
    - Проведіть обчислення, виведіть p-значення і напишіть висновок за результатами тесту.


**Формулювання гіпотез:**

**Нульова гіпотеза ($H_0$):** утримання гравця на 7-й день (retention_7) не залежить від версії гри (version). Показники утримання є незалежними від розташування воріт.

**Альтернативна гіпотеза ($H_1$):** існує залежність між версією гри (version) та утриманням гравця на 7-й день (retention_7).

In [16]:
import pandas as pd
from scipy.stats import chi2_contingency

# Побудова таблиці спряженості (contingency table)
contingency_table = pd.crosstab(df['version'], df['retention_7'])

# Проведення тесту Хі-квадрат
chi2, p_val, dof, expected = chi2_contingency(contingency_table)

print(f"Chi2 statistic: {chi2:.4f}")
print(f"p-value: {p_val:.4f}")

Chi2 statistic: 9.9591
p-value: 0.0016


**Висновок:**

Оскільки p-значення ($p = 0.0016$) є меншим за обраний рівень значущості $\alpha = 0.05$ (5%), ми відхиляємо нульову гіпотезу ($H_0$). Це підтверджує наявність статистично значущої залежності між розташуванням воріт у грі та утриманням користувачів на 7-й день. Залишення воріт на 30-му рівні забезпечує краще утримання гравців, ніж їх перенесення на 40-вий рівень.